# Silver Layer: Clean and Transform Amazon Products Data

This notebook reads from the bronze layer, applies data quality rules, performs transformations, and loads into the silver layer.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, current_timestamp, lit, trim, upper, lower, regexp_replace,
    when, isnan, isnull, length, concat_ws, to_date, cast, count
)
from datetime import datetime

print("=" * 80)
print("SILVER LAYER TRANSFORMATION: Amazon Products")
print("=" * 80)
print(f"Start Time: {datetime.now()}")
print()

In [ ]:
# Configuration
source_table = "bronze_amazon_products"
target_table = "silver_amazon_products"

print(f"Source: {source_table}")
print(f"Target: {target_table}")
print()

In [ ]:
# Read from Bronze Layer
print(f"Reading from bronze layer: {source_table}")
try:
    df_bronze = spark.table(source_table)
    bronze_count = df_bronze.count()
    print(f"✅ Read {bronze_count} rows from bronze layer")
    print()
except Exception as e:
    print(f"❌ Error reading from bronze layer: {str(e)}")
    raise

In [ ]:
# Display Source Data
print("Bronze Layer Schema:")
print("-" * 80)
df_bronze.printSchema()
print()

print("Bronze Layer Sample (first 5 rows):")
print("-" * 80)
df_bronze.show(5, truncate=False)
print()

In [ ]:
# Data Quality Checks
print("Performing data quality checks...")

# Check for null values in critical columns
null_checks = df_bronze.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in df_bronze.columns]
)
print("Null value counts:")
null_checks.show()
print()

In [ ]:
# Data Cleaning and Transformation
print("Applying transformations...")

df_silver = df_bronze

# 1. Remove complete duplicates
initial_count = df_silver.count()
df_silver = df_silver.dropDuplicates()
dedup_count = df_silver.count()
print(f"  • Removed {initial_count - dedup_count} duplicate rows")

# 2. Trim whitespace from string columns
string_columns = [field.name for field in df_silver.schema.fields
                  if str(field.dataType) == "StringType"]
for col_name in string_columns:
    df_silver = df_silver.withColumn(col_name, trim(col(col_name)))
print(f"  • Trimmed whitespace from {len(string_columns)} string columns")

# 3. Standardize text fields (example: uppercase category if it exists)
if "category" in df_silver.columns:
    df_silver = df_silver.withColumn("category", upper(col("category")))
    print(f"  • Standardized category to uppercase")

# 4. Handle null values (replace with appropriate defaults or filter)
# Example: Remove rows where critical ID column is null
if "product_id" in df_silver.columns:
    before_filter = df_silver.count()
    df_silver = df_silver.filter(col("product_id").isNotNull())
    after_filter = df_silver.count()
    print(f"  • Removed {before_filter - after_filter} rows with null product_id")

# 5. Add silver layer metadata columns
df_silver = df_silver \
    .withColumn("silver_processing_timestamp", current_timestamp()) \
    .withColumn("data_quality_flag", lit("VALID")) \
    .withColumn("silver_layer_version", lit("1.0"))

print(f"  • Added silver layer metadata columns")
print()

In [ ]:
# Data Validation
print("Validating transformed data...")
silver_count = df_silver.count()
print(f"  • Total rows after transformation: {silver_count}")
print(f"  • Data quality pass rate: {(silver_count/bronze_count)*100:.2f}%")
print()

In [ ]:
# Display Transformed Data
print("Silver Layer Schema:")
print("-" * 80)
df_silver.printSchema()
print()

print("Silver Layer Sample (first 5 rows):")
print("-" * 80)
df_silver.show(5, truncate=False)
print()

In [ ]:
# Write to Silver Layer
print(f"Writing to silver layer: {target_table}")
try:
    df_silver.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target_table)

    print(f"✅ Successfully wrote {silver_count} rows to {target_table}")
    print()

except Exception as e:
    print(f"❌ Error writing to silver layer: {str(e)}")
    raise

In [ ]:
# Verification
print("Verifying data in silver layer...")
df_verify = spark.table(target_table)
verify_count = df_verify.count()
print(f"✅ Verified: {verify_count} rows in {target_table}")
print()

In [ ]:
# Summary
print("=" * 80)
print("SILVER LAYER TRANSFORMATION COMPLETE")
print("=" * 80)
print(f"Source Table: {source_table}")
print(f"Target Table: {target_table}")
print(f"Input Rows: {bronze_count}")
print(f"Output Rows: {silver_count}")
print(f"Records Filtered: {bronze_count - silver_count}")
print(f"Data Quality Rate: {(silver_count/bronze_count)*100:.2f}%")
print(f"End Time: {datetime.now()}")
print()